In [ ]:
!pip install -r requirements.txt

In [ ]:
import json
from pprint import pprint

import boto3
from mypy_boto3_bedrock_runtime.client import BedrockRuntimeClient
from mypy_boto3_bedrock_agent.client import AgentsforBedrockClient
from mypy_boto3_bedrock_agent_runtime.client import AgentsforBedrockRuntimeClient


# Your own KB and data source. You can find ids from the management console.
knowledge_base_id = "xxx"
data_source_id = "xxx"

# LLM
model_id = "anthropic.claude-3-haiku-20240307-v1:0"
region = "ap-northeast-1"

agents_client: "AgentsforBedrockClient" = boto3.client(
    "bedrock-agent", region_name=region)
agents_runtime_client: "AgentsforBedrockRuntimeClient" = boto3.client(
    "bedrock-agent-runtime", region_name=region)
bedrock_runtime_client: "BedrockRuntimeClient" = boto3.client(
    service_name="bedrock-runtime", region_name=region)

## Retrieval

Document of `retrieve`
https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agent-runtime/client/retrieve.html

In [ ]:
query = "Tokyo population"
response = agents_runtime_client.retrieve(
    knowledgeBaseId=knowledge_base_id,
    retrievalConfiguration={
        "vectorSearchConfiguration": {
            "overrideSearchType": "HYBRID",
            "numberOfResults": 3
        }
    },
    retrievalQuery={
        "text": query
    }
)
pprint(response)

In [ ]:
prompt_template = \
"""
The section enclosed in <context></context> below contains a list of search results that are considered relevant to the user's query.  
Please read it carefully.

<context>
{context}
</context>

You are a helpful AI assistant. You will answer the user's question sincerely based on the information provided within <context></context>.  
However, if the answer to the question is **not** written in the <context></context>, please honestly respond with: “I don’t know.”

The section enclosed in <question></question> below contains the user’s question.

<question>
{question}
</question>

Please answer the user’s question.

Before providing your answer, describe your reasoning process inside the <thinking></thinking> tag, and then include your final answer within the <answer></answer> tag.
"""

In [ ]:
def invoke_llm(prompt: str, model_id: str, bedrock_runtime_client) -> str:
    response = bedrock_runtime_client.converse(
        modelId=model_id,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "text": prompt
                    }
                ],
            }
        ],
        inferenceConfig={
            "temperature": 0.0
        }
    )
    result = response['output']['message']['content'][0]['text']
    return result

In [ ]:
def retrieve_context(
    query: str,
    knowledge_base_id: str,
    agents_runtime_client: "AgentsforBedrockRuntimeClient"
) -> list[dict]:
    """Perform a search against the knowledge base to retrieve documents that are relevant to the user’s query.
    """
    response = agents_runtime_client.retrieve(
        knowledgeBaseId=knowledge_base_id,
        retrievalConfiguration={
            "vectorSearchConfiguration": {
                "overrideSearchType": "HYBRID",
                "numberOfResults": 3
            }
        },
        retrievalQuery={
            "text": query
        }
    )
    return response["retrievalResults"]

In [ ]:
def ask_question_naive_rag(
    question: str,
    knowledge_base_id: str,
    agents_runtime_client: "AgentsforBedrockRuntimeClient",
    bedrock_runtime_client: "BedrockRuntimeClient"
) -> str:
    """Achieve the simplest workflow: Question → Search → LLM → Answer.
    """
    context = retrieve_context(question, knowledge_base_id, agents_runtime_client)
    prompt = prompt_template.format(
        context=json.dumps(context),
        question=question
    )
    llm_response = invoke_llm(prompt, model_id, bedrock_runtime_client)
    print("retrieved context")
    print("="*30)
    print("\n".join(_context["content"]["text"] for _context in context))
    print("="*30)
    return llm_response

In [ ]:
%%time
answer = ask_question_naive_rag(
    "Please tell me the list of subway lines in Tokyo.", knowledge_base_id,
    agents_runtime_client,
    bedrock_runtime_client
)
print(answer)

### Hosting your RAG application with Streamlit

Configuration of env variables

In [ ]:
!echo "KNOWLEDGE_BASE_ID=$knowledge_base_id" > .env
!echo "DATA_SOURCE_ID=$data_source_id" >> .env
!cat .env

In [ ]:
!streamlit run app/app.py  --server.baseUrlPath="/proxy/absolute/8501"